# EDA - MEVO GBFS

Source: https://gbfs.urbansharing.com/rowermevo.pl/gbfs.json

`gbfs.json` is an endpoint catalog - it lists the feeds (tables) together with their URLs. Each feed is fetched separately and converted into a DataFrame.

In [ ]:
import pandas as pd
import requests

GBFS_URL = "https://gbfs.urbansharing.com/rowermevo.pl/gbfs.json"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)


def fetch_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def show_table(name: str, df: pd.DataFrame, n: int = 10) -> None:
    print(f"Table '{name}': {df.shape[0]} rows, {df.shape[1]} columns")

    print("\nColumn types and null counts:")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "null_count": df.isna().sum()}))

    print(f"\nFirst {min(n, len(df))} rows:")
    display(df.head(n))

    # Columns holding lists/dicts are unhashable, so describe() needs them cast to strings.
    df_desc = df.apply(
        lambda col: col.astype(str)
        if col.map(lambda x: isinstance(x, (list, dict))).any()
        else col
    )
    print("\nDescriptive statistics (describe) - all columns:")
    display(df_desc.describe(include="all").T)

## Endpoints listed in `gbfs.json`

In [38]:
gbfs = fetch_json(GBFS_URL)

feeds = pd.DataFrame(
    [
        {"language": lang, "name": feed["name"], "url": feed["url"]}
        for lang, block in gbfs["data"].items()
        for feed in block["feeds"]
    ]
)

print(f"GBFS version: {gbfs['version']}, refresh interval (ttl): {gbfs['ttl']} s")
print(f"Available endpoints ({len(feeds)}):")
display(feeds)

GBFS version: 2.3, refresh interval (ttl): 15 s
Available endpoints (6):


,language,name,url
0,pl,system_information,https://gbfs.urbansharing.com/rowermevo.pl/system_information.json
1,pl,vehicle_types,https://gbfs.urbansharing.com/rowermevo.pl/vehicle_types.json
2,pl,system_pricing_plans,https://gbfs.urbansharing.com/rowermevo.pl/system_pricing_plans.json
3,pl,station_information,https://gbfs.urbansharing.com/rowermevo.pl/station_information.json
4,pl,station_status,https://gbfs.urbansharing.com/rowermevo.pl/station_status.json
5,pl,free_bike_status,https://gbfs.urbansharing.com/rowermevo.pl/free_bike_status.json


In [39]:
# Key inside the 'data' section that holds the feed's list of records.
RECORDS_KEY = {
    "vehicle_types": "vehicle_types",
    "system_pricing_plans": "plans",
    "station_information": "stations",
    "station_status": "stations",
    "free_bike_status": "bikes",
}

tables = {}
for feed in feeds.itertuples():
    data = fetch_json(feed.url)["data"]
    if feed.name in RECORDS_KEY:
        tables[feed.name] = pd.json_normalize(data[RECORDS_KEY[feed.name]])
    else:
        tables[feed.name] = pd.json_normalize(data)

print("Row and column counts of the downloaded tables:")
display(
    pd.DataFrame(
        [{"table": name, "rows": df.shape[0], "columns": df.shape[1]} for name, df in tables.items()]
    )
)

Row and column counts of the downloaded tables:


,table,rows,columns
0,system_information,1,11
1,vehicle_types,2,7
2,system_pricing_plans,2,7
3,station_information,865,12
4,station_status,865,9
5,free_bike_status,891,10


## `system_information`
Basic system information: name, operator, timezone, contact details, app links. A single record.

In [40]:
print("system_information record (transposed for readability):")
display(tables["system_information"].T.rename(columns={0: "value"}))

system_information record (transposed for readability):


,value
system_id,inurba-gdansk
language,pl
name,MEVO
operator,Inurba
timezone,Europe/Warsaw
phone_number,+48587391123
email,kontakt@rowermevo.pl
rental_apps.android.discovery_uri,rowermevo://
rental_apps.android.store_uri,https://play.google.com/store/apps/details?id=com.urbansharing.citybike.gdansk
rental_apps.ios.discovery_uri,rowermevo://


## `vehicle_types`
Vehicle type dictionary (e.g. regular bike / e-bike): propulsion type, form factor, range. One row = one vehicle type.

In [41]:
show_table("vehicle_types", tables["vehicle_types"])

Table 'vehicle_types': 2 rows, 7 columns

Column types and null counts:


,dtype,null_count
vehicle_type_id,str,0
form_factor,str,0
default_pricing_plan_id,str,0
propulsion_type,str,0
name,str,0
rider_capacity,int64,0
max_range_meters,float64,1



First 2 rows:


,vehicle_type_id,form_factor,default_pricing_plan_id,propulsion_type,name,rider_capacity,max_range_meters
0,bike,bicycle,bike,human,Traditional bike,1,NaN
1,ebike,bicycle,ebike,electric_assist,Electric bike,1,100000.0



Descriptive statistics (describe) – all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
vehicle_type_id,2,2,bike,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
form_factor,2,1,bicycle,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default_pricing_plan_id,2,2,bike,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
propulsion_type,2,2,human,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,2,2,Traditional bike,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rider_capacity,2.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
max_range_meters,1.0,NaN,NaN,NaN,100000.0,NaN,100000.0,100000.0,100000.0,100000.0,100000.0


## `system_pricing_plans`
Pricing plans: name, currency, base price, description, per-minute pricing (`per_min_pricing` - a list). One row = one plan.

In [42]:
show_table("system_pricing_plans", tables["system_pricing_plans"])

Table 'system_pricing_plans': 2 rows, 7 columns

Column types and null counts:


,dtype,null_count
plan_id,str,0
name,str,0
currency,str,0
price,int64,0
description,str,0
is_taxable,bool,0
per_min_pricing,object,0



First 2 rows:


,plan_id,name,currency,price,description,is_taxable,per_min_pricing
0,bike,Traditional bike charge,PLN,0,Cheapest subscription with the system for a single trip. Visit our websites ...,False,"[{'start': 1, 'rate': 0.15, 'end': 720, 'interval': 1}]"
1,ebike,Ebike charge,PLN,0,Cheapest subscription with the system for a single trip. Visit our websites ...,False,"[{'start': 1, 'rate': 0.3, 'end': 720, 'interval': 1}]"



Descriptive statistics (describe) – all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
plan_id,2,2,bike,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,2,2,Traditional bike charge,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
currency,2,1,PLN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,2.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
description,2,1,Cheapest subscription with the system for a single trip. Visit our websites ...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_taxable,2,1,False,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
per_min_pricing,2,2,"[{'start': 1, 'rate': 0.15, 'end': 720, 'interval': 1}]",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `station_information`
Static station data: id, name, address, coordinates, capacity, station area. One row = one station.

In [43]:
show_table("station_information", tables["station_information"])

Table 'station_information': 865 rows, 12 columns

Column types and null counts:


,dtype,null_count
station_id,str,0
name,str,0
address,str,0
cross_street,str,0
lat,float64,0
lon,float64,0
is_virtual_station,bool,0
capacity,int64,0
station_area.type,str,0
station_area.coordinates,object,0



First 10 rows:


,station_id,name,address,cross_street,lat,lon,is_virtual_station,capacity,station_area.type,station_area.coordinates,rental_uris.android,rental_uris.ios
0,8411,KAR019,"Osiedle Wybickiego 33, 83-300 Kartuzy","Osiedle Wybickiego 33, 83-300 Kartuzy",54.32505,18.19296,True,10,MultiPolygon,"[[[[18.19296, 54.325319796109106], [18.192801772169656, 54.325303525309344],...",rowermevo://stations/8411,rowermevo://stations/8411
1,8409,GDY182,"Starowiejska 48, 81-356 Gdynia","Starowiejska 48, 81-356 Gdynia",54.52115,18.53404,True,10,MultiPolygon,"[[[[18.53404, 54.52141979610912], [18.533881013273827, 54.52140352530859], [...",rowermevo://stations/8409,rowermevo://stations/8409
2,8407,GDY178,"Morska 49, 81-323 Gdynia","Morska 49, 81-323 Gdynia",54.52290,18.52172,True,10,MultiPolygon,"[[[[18.52172, 54.52316979610911], [18.521561006460256, 54.52315352530859], [...",rowermevo://stations/8407,rowermevo://stations/8407
3,8404,GDY170,"Aleja Zwycięstwa 245, 81-402 Gdynia","Aleja Zwycięstwa 245, 81-402 Gdynia",54.47962,18.55282,True,10,MultiPolygon,"[[[[18.55282, 54.47988979610911], [18.552661174754967, 54.479873525308754], ...",rowermevo://stations/8404,rowermevo://stations/8404
4,8403,GDY167,"Plażowa 12, 81-523 Gdynia","Plażowa 12, 81-523 Gdynia",54.47698,18.56204,True,10,MultiPolygon,"[[[[18.56204, 54.47724979610911], [18.561881185006175, 54.47723352530876], [...",rowermevo://stations/8403,rowermevo://stations/8403
5,8400,GDY164,"Juliana Ejsmonda, Gdynia","Juliana Ejsmonda, Gdynia",54.50262,18.55764,True,10,MultiPolygon,"[[[[18.55764, 54.502889796109116], [18.557481085374913, 54.50287352530867], ...",rowermevo://stations/8400,rowermevo://stations/8400
6,8399,GDY184,"Rumska 8, 81-074 Gdynia","Rumska 8, 81-074 Gdynia",54.53826,18.44857,True,50,MultiPolygon,"[[[[18.44857, 54.53852979610912], [18.448410946625124, 54.538513525308524], ...",rowermevo://stations/8399,rowermevo://stations/8399
7,8398,GDY176,"Swarzewska 47, 81-059 Gdynia","Swarzewska 47, 81-059 Gdynia",54.53797,18.46359,True,10,MultiPolygon,"[[[[18.46359, 54.53823979610912], [18.46343094775535, 54.53822352530854], [1...",rowermevo://stations/8398,rowermevo://stations/8398
8,8397,GDY175,"plac Konstytucji 5, 81-001 Gdynia","plac Konstytucji 5, 81-001 Gdynia",54.52235,18.53082,True,10,MultiPolygon,"[[[[18.530820000000002, 54.52261979610911], [18.53066100860174, 54.522603525...",rowermevo://stations/8397,rowermevo://stations/8397
9,8396,GDY173,"Śląska 33, 81-310 Gdynia","Śląska 33, 81-310 Gdynia",54.51537,18.53258,True,10,MultiPolygon,"[[[[18.532579999999996, 54.51563979610912], [18.532421035772874, 54.51562352...",rowermevo://stations/8396,rowermevo://stations/8396



Descriptive statistics (describe) – all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
station_id,865,865,8411,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
name,865,865,KAR019,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
address,865,859,Lawendowe wzgórze,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cross_street,865,860,Rogalińska 21,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,865.0,NaN,NaN,NaN,54.412504,0.131579,54.051165,54.346008,54.390708,54.499045,54.833472
lon,865.0,NaN,NaN,NaN,18.549382,0.124623,17.79401,18.497385,18.57222,18.61819,18.933902
is_virtual_station,865,1,True,865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
capacity,865.0,NaN,NaN,NaN,12.867052,37.894281,7.0,10.0,10.0,10.0,1000.0
station_area.type,865,1,MultiPolygon,865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
station_area.coordinates,865,865,"[[[[18.19296, 54.325319796109106], [18.192801772169656, 54.325303525309344],...",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `station_status`
Current station state: number of available bikes and free docks, operational flags, last report time, availability by vehicle type (`vehicle_types_available` - a list). One row = one station at fetch time.

In [44]:
show_table("station_status", tables["station_status"])

Table 'station_status': 865 rows, 9 columns

Column types and null counts:


,dtype,null_count
station_id,str,0
is_installed,bool,0
is_renting,bool,0
is_returning,bool,0
last_reported,int64,0
num_vehicles_available,int64,0
num_bikes_available,int64,0
num_docks_available,int64,0
vehicle_types_available,object,0



First 10 rows:


,station_id,is_installed,is_renting,is_returning,last_reported,num_vehicles_available,num_bikes_available,num_docks_available,vehicle_types_available
0,8411,True,True,True,1791453258,0,0,10,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."
1,8409,True,True,True,1791453258,1,1,9,"[{'vehicle_type_id': 'bike', 'count': 1}, {'vehicle_type_id': 'ebike', 'coun..."
2,8407,True,True,True,1791453258,0,0,10,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."
3,8404,True,True,True,1791453258,2,2,7,"[{'vehicle_type_id': 'bike', 'count': 1}, {'vehicle_type_id': 'ebike', 'coun..."
4,8403,True,True,True,1791453258,4,4,6,"[{'vehicle_type_id': 'bike', 'count': 4}, {'vehicle_type_id': 'ebike', 'coun..."
5,8400,True,True,True,1791453258,1,1,9,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."
6,8399,True,True,True,1791453258,0,0,49,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."
7,8398,True,True,True,1791453258,5,5,5,"[{'vehicle_type_id': 'bike', 'count': 1}, {'vehicle_type_id': 'ebike', 'coun..."
8,8397,True,True,True,1791453258,0,0,10,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."
9,8396,True,True,True,1791453258,0,0,10,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun..."



Descriptive statistics (describe) – all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
station_id,865,865,8411,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_installed,865,1,True,865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_renting,865,1,True,865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_returning,865,1,True,865,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_reported,865.0,NaN,NaN,NaN,1791453258.0,0.0,1791453258.0,1791453258.0,1791453258.0,1791453258.0,1791453258.0
num_vehicles_available,865.0,NaN,NaN,NaN,3.787283,5.975147,0.0,1.0,2.0,5.0,61.0
num_bikes_available,865.0,NaN,NaN,NaN,3.787283,5.975147,0.0,1.0,2.0,5.0,61.0
num_docks_available,865.0,NaN,NaN,NaN,9.22659,37.640441,0.0,5.0,8.0,9.0,997.0
vehicle_types_available,865,119,"[{'vehicle_type_id': 'bike', 'count': 0}, {'vehicle_type_id': 'ebike', 'coun...",207,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## `free_bike_status`
Bikes available outside stations: id, coordinates, vehicle type, reserved/disabled flags. One row = one bike at fetch time.

In [45]:
show_table("free_bike_status", tables["free_bike_status"])

Table 'free_bike_status': 891 rows, 10 columns

Column types and null counts:


,dtype,null_count
bike_id,str,0
lat,float64,0
lon,float64,0
is_reserved,bool,0
is_disabled,bool,0
vehicle_type_id,str,0
last_reported,int64,0
current_range_meters,float64,329
rental_uris.android,str,0
rental_uris.ios,str,0



First 10 rows:


,bike_id,lat,lon,is_reserved,is_disabled,vehicle_type_id,last_reported,current_range_meters,rental_uris.android,rental_uris.ios
0,vloc_1QAh6KoXzQPOKmi757GtHQNQ,54.525450,18.427425,False,False,bike,1791453258,NaN,rowermevo://vehicles/vloc_1QAh6KoXzQPOKmi757GtHQNQ,rowermevo://vehicles/vloc_1QAh6KoXzQPOKmi757GtHQNQ
1,vloc_1XMshRJfREtbpUlMeLJtGHuO,54.530482,18.483083,False,True,ebike,1791453258,61000.0,rowermevo://vehicles/vloc_1XMshRJfREtbpUlMeLJtGHuO,rowermevo://vehicles/vloc_1XMshRJfREtbpUlMeLJtGHuO
2,vloc_1MvMw5tyVlsleDYT5ISbRfcP,54.535713,18.447635,False,True,ebike,1791453258,NaN,rowermevo://vehicles/vloc_1MvMw5tyVlsleDYT5ISbRfcP,rowermevo://vehicles/vloc_1MvMw5tyVlsleDYT5ISbRfcP
3,vloc_1G6tl4s22A6losqWGuY8Vn34,54.348785,18.655918,False,False,ebike,1791453258,40000.0,rowermevo://vehicles/vloc_1G6tl4s22A6losqWGuY8Vn34,rowermevo://vehicles/vloc_1G6tl4s22A6losqWGuY8Vn34
4,vloc_1RfpSELE28VVyMLou9ACUn3l,54.540385,18.477890,False,False,ebike,1791453258,71000.0,rowermevo://vehicles/vloc_1RfpSELE28VVyMLou9ACUn3l,rowermevo://vehicles/vloc_1RfpSELE28VVyMLou9ACUn3l
5,vloc_13jzE3Dou6GaDLl9I5L1b8xh,54.613695,18.488630,False,True,ebike,1791453258,NaN,rowermevo://vehicles/vloc_13jzE3Dou6GaDLl9I5L1b8xh,rowermevo://vehicles/vloc_13jzE3Dou6GaDLl9I5L1b8xh
6,vloc_15JNE2Ag5HsRwPfyNFwAw5hr,54.270903,18.418122,False,True,ebike,1791453258,NaN,rowermevo://vehicles/vloc_15JNE2Ag5HsRwPfyNFwAw5hr,rowermevo://vehicles/vloc_15JNE2Ag5HsRwPfyNFwAw5hr
7,vloc_1HbMOeODKO32v8s1tgepkHrY,54.068992,18.718945,False,False,ebike,1791453258,100000.0,rowermevo://vehicles/vloc_1HbMOeODKO32v8s1tgepkHrY,rowermevo://vehicles/vloc_1HbMOeODKO32v8s1tgepkHrY
8,vloc_18vPLgzxTxmYrFSOcQyx6wvJ,54.361062,18.647220,False,True,ebike,1791453258,NaN,rowermevo://vehicles/vloc_18vPLgzxTxmYrFSOcQyx6wvJ,rowermevo://vehicles/vloc_18vPLgzxTxmYrFSOcQyx6wvJ
9,vloc_1SaxM1h0sRSmhbsCBm922ZLi,54.359943,18.735280,False,True,ebike,1791453258,1000.0,rowermevo://vehicles/vloc_1SaxM1h0sRSmhbsCBm922ZLi,rowermevo://vehicles/vloc_1SaxM1h0sRSmhbsCBm922ZLi



Descriptive statistics (describe) – all columns:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
bike_id,891,891,vloc_1QAh6KoXzQPOKmi757GtHQNQ,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,891.0,NaN,NaN,NaN,54.414533,0.296735,54.041038,54.338825,54.381968,54.474097,60.248225
lon,891.0,NaN,NaN,NaN,18.53857,0.372445,10.72688,18.478037,18.561145,18.622728,24.842288
is_reserved,891,1,False,891,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_disabled,891,2,False,498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vehicle_type_id,891,2,ebike,781,NaN,NaN,NaN,NaN,NaN,NaN,NaN
last_reported,891.0,NaN,NaN,NaN,1791453258.0,0.0,1791453258.0,1791453258.0,1791453258.0,1791453258.0,1791453258.0
current_range_meters,562.0,NaN,NaN,NaN,39222.419929,32700.368156,1000.0,1250.0,35000.0,65000.0,100000.0
rental_uris.android,891,891,rowermevo://vehicles/vloc_1QAh6KoXzQPOKmi757GtHQNQ,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rental_uris.ios,891,891,rowermevo://vehicles/vloc_1QAh6KoXzQPOKmi757GtHQNQ,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
